# 05_Face Machine Learning

## Objective
The objective of this notebook is to train and evaluate face recognition models using extracted features.

## Method
Supervised machine learning training and evaluation workflow.

## Input
Face feature dataset from the extraction stage.

## Processing Tasks

1. – Load & Validate Dataset.
2. – Dataset Preparation.
3. – Train-Test Split.
4. – Feature Scaling.
5. – Model Training.
6. – SVM Model Training.
7. – Model Evaluation.
8. – Classification Metrics.
9. – Confusion Matrix.
10. – Save Trained SVM Model.
11. small test.

## Output
- Face model performance metrics.
- Model comparison results.
- Saved best face recognition model.

## Purpose
Build a reliable face recognition model for use in the verification pipeline.


In [ ]:
# Import Required Libraries

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns


# Configure Project Paths

PROJECT_PATH = Path(
    r"E:\Multimodal Biometric Verification System"
)

FEATURES_PATH = (
    PROJECT_PATH
    / "Dataset"
    / "Processed"
    / "Face"
    / "Face_Features"
)

COMBINED_FEATURES_FILE = (
    FEATURES_PATH / "Face_Combined_Features.csv"
)


# Configure Dataset Parameters

TARGET_COLUMN = "User_ID"
METADATA_COLUMNS = ["Image_Name"]

RANDOM_STATE = 42
TEST_SIZE = 0.20


print("Environment Configuration Completed")
print("-" * 45)

print("Project Path:", PROJECT_PATH)
print("Features Path:", FEATURES_PATH)
print("Dataset File:", COMBINED_FEATURES_FILE)

# 1. – Load & Validate Dataset


In [ ]:
# Load Combined Face Features Dataset

face_df = pd.read_csv(COMBINED_FEATURES_FILE)


# Display Dataset Information

print("Face Dataset Loaded Successfully")
print("-" * 45)

print("Dataset Shape:", face_df.shape)

print("\nFirst 10 Columns:")
print(face_df.columns[:10].tolist())

print("\nLast 5 Columns:")
print(face_df.columns[-5:].tolist())

print("\nIdentity Distribution:")
print(face_df[TARGET_COLUMN].value_counts())

print("\nMissing Values:", face_df.isnull().sum().sum())

print(
    "\nInvalid Numerical Values:",
    not np.isfinite(
        face_df.select_dtypes(include=np.number).to_numpy()
    ).all()
)

# 2. – Dataset Preparation

This section separates the input features (X) from the target labels (y).

- X contains the combined HOG and LBP numerical features.
- y contains the User_ID labels.


In [ ]:
# Separate Features and Target

X = face_df.drop(
    columns=[TARGET_COLUMN, *METADATA_COLUMNS]
)

y = face_df[TARGET_COLUMN]


# Validate Prepared Dataset

print("Dataset Preparation Completed")
print("-" * 45)

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

print("\nFeature Count:", X.shape[1])

print("\nTarget Distribution:")
print(y.value_counts())

print("\nFeature Data Type:")
print(X.dtypes.value_counts())

# 3. – Train-Test Split

This section splits the dataset into training and testing subsets.


In [ ]:
# Split Dataset into Training and Testing Sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)


# Display Split Information

print("Train-Test Split Completed")
print("-" * 45)

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("\nTraining Target Distribution:")
print(y_train.value_counts())

print("\nTesting Target Distribution:")
print(y_test.value_counts())

# 4. – Feature Scaling


In [ ]:
# Configure SVM Feature Scaling Pipeline

svm_scaler = StandardScaler()

X_train_svm = svm_scaler.fit_transform(X_train)

X_test_svm = svm_scaler.transform(X_test)


# Validate Scaled Data

print("Feature Scaling Completed")
print("-" * 45)

print("Training Shape:", X_train_svm.shape)
print("Testing Shape:", X_test_svm.shape)

print("\nTraining Mean:", np.mean(X_train_svm))
print("Training Standard Deviation:", np.std(X_train_svm))

print("\nTesting Mean:", np.mean(X_test_svm))
print("Testing Standard Deviation:", np.std(X_test_svm))

# 5. – Model Training

This section trains the Random Forest classifier using the combined HOG and LBP features.


In [ ]:
# Initialize Random Forest Model

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1
)


# Train Random Forest Model

rf_model.fit(X_train, y_train)


print("Random Forest Training Completed")
print("-" * 45)

print("Number of Trees:", rf_model.n_estimators)
print("Training Samples:", X_train.shape[0])
print("Number of Features:", X_train.shape[1])

# 6. – SVM Model Training


In [ ]:
# Initialize SVM Model

svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,
    random_state=RANDOM_STATE
)


# Train SVM Model

svm_model.fit(X_train_svm, y_train)


print("SVM Training Completed")
print("-" * 45)

print("Kernel:", svm_model.kernel)
print("C:", svm_model.C)
print("Gamma:", svm_model.gamma)
print("Training Samples:", X_train_svm.shape[0])
print("Number of Features:", X_train_svm.shape[1])

# 7. – Model Evaluation

This section evaluates the trained Random Forest and SVM classifiers using the unseen testing dataset.


In [ ]:
# Generate Predictions

rf_predictions = rf_model.predict(X_test)

svm_predictions = svm_model.predict(X_test_svm)


# Display Predictions

print("Model Predictions Completed")
print("-" * 45)

print("Random Forest Predictions:")
print(rf_predictions)

print("\nSVM Predictions:")
print(svm_predictions)

print("\nActual Labels:")
print(y_test.to_numpy())

# 8. – Classification Metrics

This section calculates the classification performance metrics for both Random Forest and SVM.


In [ ]:
# Calculate Evaluation Metrics

rf_metrics = {
    "Accuracy": accuracy_score(y_test, rf_predictions),
    "Precision": precision_score(
        y_test, rf_predictions, average="weighted", zero_division=0
    ),
    "Recall": recall_score(
        y_test, rf_predictions, average="weighted", zero_division=0
    ),
    "F1-Score": f1_score(
        y_test, rf_predictions, average="weighted", zero_division=0
    )
}

svm_metrics = {
    "Accuracy": accuracy_score(y_test, svm_predictions),
    "Precision": precision_score(
        y_test, svm_predictions, average="weighted", zero_division=0
    ),
    "Recall": recall_score(
        y_test, svm_predictions, average="weighted", zero_division=0
    ),
    "F1-Score": f1_score(
        y_test, svm_predictions, average="weighted", zero_division=0
    )
}


# Create Model Comparison DataFrame

metrics_df = pd.DataFrame(
    [rf_metrics, svm_metrics],
    index=["Random Forest", "SVM"]
)


print("Model Evaluation Results")
print("-" * 45)

print(metrics_df.round(4))

# 9. – Confusion Matrix

This section visualizes the confusion matrices of the Random Forest and SVM models.


In [ ]:
rf_cm = confusion_matrix(y_test, rf_predictions)

svm_cm = confusion_matrix(y_test, svm_predictions)

# Display Random Forest Confusion Matrix
print("Random Forest Confusion Matrix")
print("-" * 40)

rf_cm_df = pd.DataFrame(
    rf_cm,
    index=rf_model.classes_,
    columns=rf_model.classes_
)

print(rf_cm_df)


# Display SVM Confusion Matrix
print("\nSVM Confusion Matrix")
print("-" * 40)

svm_cm_df = pd.DataFrame(
    svm_cm,
    index=svm_model.classes_,
    columns=svm_model.classes_
)

print(svm_cm_df)


# Visualize Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(
    rf_cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=rf_model.classes_,
    yticklabels=rf_model.classes_,
    ax=axes[0]
)

axes[0].set_title("Random Forest Confusion Matrix")
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("Actual Label")

sns.heatmap(
    svm_cm,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=svm_model.classes_,
    yticklabels=svm_model.classes_,
    ax=axes[1]
)

axes[1].set_title("SVM Confusion Matrix")
axes[1].set_xlabel("Predicted Label")
axes[1].set_ylabel("Actual Label")

plt.tight_layout()
plt.show()

# 10. – Save Trained SVM Model


In [ ]:
import joblib

MODEL_PATH = (
    PROJECT_PATH / "Models" / "Face"
)

MODEL_PATH.mkdir(parents=True, exist_ok=True)

MODEL_FILE = MODEL_PATH / "SVM_Face_Recognition.joblib"

In [ ]:
model_artifacts = {
    "model": svm_model,
    "scaler": svm_scaler,
    "feature_columns": X.columns.tolist()
}

joblib.dump(model_artifacts, MODEL_FILE)

print("SVM Model saved successfully.")
print(f"Saved Location: {MODEL_FILE}")

#### 11. small test


In [ ]:
loaded_artifacts = joblib.load(MODEL_FILE)

loaded_svm = loaded_artifacts["model"]
loaded_scaler = loaded_artifacts["scaler"]
loaded_feature_columns = loaded_artifacts["feature_columns"]

print("Model loaded successfully.")
print(f"Model Type: {type(loaded_svm).__name__}")
print(f"Scaler Type: {type(loaded_scaler).__name__}")
print(f"Number of Features: {len(loaded_feature_columns)}")